# 10 — Model Selection and Validation
### The Test Set Is Used Once

---

> **Position in the series:**
> `01_statistical_learning_theory` → `02_pac_and_vc_dimensions` → `03_linear_algebra_and_geometry` → `04_pca` → `05_gaussian_mixture_models` → `06_linear_regression` → `07_logistic_regression` → `08_svm_and_kernel_methods` → `09_decision_trees_random_forest_boosting` → **`10_model_selection_and_validation`**

---

## References

| # | Source |
|---|--------|
| [1] | T. Hastie, R. Tibshirani & J. Friedman — *The Elements of Statistical Learning*, 2nd ed., Springer, 2009 — §7.10 (Cross-Validation), §7.10.2 (The Wrong and Right Way to Do Cross-validation) |
| [2] | S. Shalev-Shwartz & S. Ben-David — *Understanding Machine Learning*, Cambridge, 2014 — Ch. 11 (Model Selection and Validation) |
| [3] | A. Géron — *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*, O'Reilly — Ch. 1 (Testing and Validating), Ch. 2 (End-to-End Machine Learning Project) |
| [4] | G. C. Cawley & N. L. C. Talbot — *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation*, JMLR, 11, 2010 |
| [5] | scikit-learn documentation — *Cross-validation: evaluating estimator performance*, https://scikit-learn.org/stable/modules/cross_validation.html; *Common pitfalls and recommended practices*, https://scikit-learn.org/stable/common_pitfalls.html |

---

## Structure

This closing notebook has no new model and no new theorem. Each section is a **problem** met when training and reporting a model, followed by its **solution**.

0. The Rule in One Page
1. Problem: The Training Score Flatters the Model
2. Problem: Tuning on the Test Set
3. Problem: One Validation Split Is Noisy
4. Problem: The Preprocessing Saw the Validation Data
5. Problem: The Winner's Validation Score Is Optimistic Too
6. The Whole Procedure, From Data to the Reported Number
7. Checklist and Summary

The models come from `07`, `08` and `09`; here they are taken from scikit-learn, because the subject is the procedure around them.

---
## 0. The Rule in One Page

> **The test set is used once:** after every decision has been made, to compute the number that is reported. Nothing is changed because of that number.

A score estimates the performance on new data only if the data it is computed on played **no role in building the model**. Building the model includes fitting the parameters, and also every choice made by looking at a score: hyperparameters, features, preprocessing, the model family, when to stop.

So the data get three roles, and each decision uses only its own part:

| Part | Used for | How often |
|---|---|---|
| **Training** | fitting the parameters | as often as needed |
| **Validation** (carved out of the training part, by K-fold) | choosing between candidates | as often as needed |
| **Test** | the reported number | **once** |

**The procedure.** The rest of the notebook follows this list.

1. Split the data once into training and test. Lock the test part away.
2. Using the training part only, compare the candidates by K-fold cross-validation, with all the preprocessing inside the folds.
3. Choose with a rule fixed in advance.
4. Refit the chosen candidate on the whole training part.
5. Open the test set, compute the score, report it. Stop.

The cell below does step 1 on the breast cancer data used in `09`. To make the rule concrete, the test part goes into a small object, `LockedTestSet`, that hands the data out one time and raises an error at the second request.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

BLUE, RED, TEAL, ORANGE, PURPLE = '#457B9D', '#E63946', '#2A9D8F', '#F4A261', '#6A0572'

class LockedTestSet:
    # Hands the test data out exactly once.
    def __init__(self, X, y):
        self._X, self._y, self.opened = X, y, False
    def __len__(self):
        return len(self._y)
    def open(self):
        if self.opened:
            raise RuntimeError("The test set has already been used once. To recompute, restart and run the notebook from the top.")
        self.opened = True
        return self._X, self._y

# Step 1: split once (stratified, 80/20) and lock the test part away
X_all, y_all = load_breast_cancer(return_X_y=True)
X_train, X_held, y_train, y_held = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=0)
TEST = LockedTestSet(X_held, y_held)
del X_all, y_all, X_held, y_held                    # from here on, only X_train / y_train are visible

print(f"training part: {len(y_train)} points, {X_train.shape[1]} features, fraction of class 1 = {y_train.mean():.3f}")
print(f"test part    : {len(TEST)} points, locked (opened: {TEST.opened})")

---
## 1. Problem: The Training Score Flatters the Model

**Problem.** The model was fitted to make the training score high, so that score says little about new data. A flexible model can reach 100% on the training points by memorizing them (`01`, §4.3).

**Solution.** Score the model on points it was not fitted on. The test set is locked, so we set aside a part of the *training* data: a **validation set**.

**In the cell.** Three models are fitted on 75% of the training part and scored on the remaining 25%. The first is an RBF SVM (`08`, §17) on unscaled features with a kernel far too narrow for them.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=0)

models = {
    'RBF SVM, unscaled, gamma = 0.01': SVC(C=10, gamma=0.01),
    'decision tree, fully grown'     : DecisionTreeClassifier(random_state=0),
    'RBF SVM, scaled, default gamma' : make_pipeline(StandardScaler(), SVC(C=1)),
}
print(f"{'model':<34} {'fitted on':>10} {'training acc':>13} {'validation acc':>15}")
for name, model in models.items():
    model.fit(X_fit, y_fit)
    print(f"{name:<34} {len(y_fit):>10} {model.score(X_fit, y_fit):>13.3f} {model.score(X_val, y_val):>15.3f}")
print(f"\nalways predicting the majority class: validation acc = {max(y_val.mean(), 1 - y_val.mean()):.3f}")

---
## 2. Problem: Tuning on the Test Set

**Problem.** It is tempting to try many models, score each one on the test set and keep the best. The best of many scores is partly luck, and the luck is reported as if it were skill. Each look is harmless alone; the *selection* is what breaks the estimate.

**How large is the effect?** The experiment removes all the skill: the labels are fair coin flips, independent of the features, so **every** model has true accuracy 0.5. We train $k$ small trees on random subsets of the features, score all of them on a test set of $n=200$ points, and keep the best. The best of $k$ test scores exceeds the truth by roughly $\sqrt{\ln k/(2n)}$, the term that the union bound over $k$ hypotheses produces in `02`, §2.3 (dashed curve in the figure).

**Solution.** Take every decision on validation data. Then the test score is computed for a single model that was fixed beforehand, and it is an honest estimate.